# VKOSPI 동적 위험 오버레이 — Colab 재현 노트북

VKOSPI를 **한국판 VIX**로 해석하고, 전일 종가에서 만든 수준·5일 급등 신호로 기준 포트폴리오의 KODEX200/USO 일부를 GLD로 옮깁니다.

- 입력: `vkospi_dynamic_colab_bundle.zip` (이 노트북과 함께 생성됨)
- 기본값: Colab에서는 836개 후보를 다시 탐색하고 2018–2026 잠금 구간을 검증
- 예상 시간: 런타임에 따라 약 3–6분
- 주의: 연구용 과거 시뮬레이션이며 투자 조언이 아닙니다.


## 1. 런타임 준비

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "matplotlib"],
        check=True,
    )

import json, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

print("Colab runtime:", IN_COLAB)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)


## 2. 실행 번들 불러오기

Colab이면 파일 선택창에서 **`vkospi_dynamic_colab_bundle.zip`**을 업로드하세요. 로컬 Jupyter에서는 노트북을 프로젝트 루트에 두면 현재 폴더를 그대로 사용합니다.


In [ ]:
def safe_extract(zip_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(destination)

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    bundle_name = next((name for name in uploaded if name.endswith(".zip")), None)
    if bundle_name is None:
        raise FileNotFoundError("vkospi_dynamic_colab_bundle.zip을 업로드해 주세요.")
    safe_extract(Path(bundle_name), Path("/content"))
    PROJECT_ROOT = Path("/content/RegimeDecisionTest")
else:
    PROJECT_ROOT = Path.cwd().resolve()

required = [
    "vkospi_dynamic_risk_experiment.py",
    "raw_data/VKOSPIData.csv",
    "raw_data/compass.db",
    "raw_data/krx_bond_index.csv",
    "cache/market_daily.csv",
    "results/vkospi_selected_backtest.csv",
]
missing = [name for name in required if not (PROJECT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f"실행 번들에 필요한 파일이 없습니다: {missing}")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("필수 입력 6개 확인 완료")


## 3. 전체 재탐색 및 잠금 검증

`RUN_FULL_RECALIBRATION=True`이면 2007–2017 자료만으로 528개 거친 후보와 315개 정밀 후보를 평가합니다. 2018년 이후 구간은 승자를 고르는 데 쓰지 않습니다. 빠르게 기존 산출물만 살펴보려면 `False`로 바꾸세요.


In [ ]:
RUN_FULL_RECALIBRATION = True if IN_COLAB else False

if RUN_FULL_RECALIBRATION:
    completed = subprocess.run(
        [sys.executable, "-u", "vkospi_dynamic_risk_experiment.py"],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    print(completed.stdout[-5000:])
else:
    print("재탐색을 생략하고 번들 안의 검증 결과를 읽습니다.")


## 4. 선택된 규칙과 잠금 성과

In [ ]:
RESULTS = PROJECT_ROOT / "results"
report = json.loads((RESULTS / "vkospi_dynamic_validation.json").read_text(encoding="utf-8"))
locked = report["locked"]["monthly_reference_reconciled"]

display(Markdown("### 선택된 파라미터"))
display(pd.Series(report["winner"], name="value").to_frame())

metric_names = ["CAGR", "Sharpe", "MDD", "Calmar"]
metrics = pd.DataFrame({
    "Reference": pd.Series(locked["reference"])[metric_names],
    "VKOSPI Dynamic": pd.Series(locked["candidate"])[metric_names],
    "Delta": pd.Series(locked["deltas"])[metric_names],
})
display(Markdown("### Locked 2018–2026 · 월간 기준 재조정"))
display(metrics.style.format("{:.4f}"))
print("CAGR / Sharpe / MDD 동시 개선:", locked["passes_all_three"])


### 비교 기준 메모

`monthly_reference_reconciled`는 이미 검증된 월간 기준수익에 **동적 일간 전략 / 기준 일간 재구성**의 상대수익 팩터를 곱합니다. 월간→일간 변환 오차를 오버레이 알파로 오인하지 않기 위한 권위 비교이며, 별도로 실제 일간 재구성 결과도 보고합니다.


In [ ]:
reference = pd.read_csv(RESULTS / "vkospi_selected_backtest.csv", index_col="month")
dynamic = pd.read_csv(RESULTS / "vkospi_dynamic_reconciled_monthly.csv", index_col="month")
common = reference.index.intersection(dynamic.index)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(common, reference.loc[common, "nav"], label="Reference", color="#9aa3a0", lw=2)
axes[0].plot(common, dynamic.loc[common, "nav"], label="VKOSPI Dynamic", color="#007a5e", lw=2.4)
axes[0].set_title("Cumulative NAV · monthly reconciled")
axes[0].set_ylabel("NAV")
axes[0].legend()
axes[0].grid(alpha=.2)
axes[1].fill_between(range(len(common)), 100 * reference.loc[common, "drawdown"], color="#aab2af", alpha=.35, label="Reference")
axes[1].plot(range(len(common)), 100 * dynamic.loc[common, "drawdown"], color="#007a5e", lw=2, label="VKOSPI Dynamic")
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_xticks(range(0, len(common), 24), common[::24], rotation=45)
axes[1].grid(alpha=.2)
axes[1].legend()
plt.tight_layout()
plt.show()


## 5. 실제 신호와 위험 이전

In [ ]:
daily = pd.read_csv(RESULTS / "vkospi_dynamic_daily.csv", parse_dates=["date", "signal_date"])
locked_daily = daily.loc[daily["date"] >= "2018-01-01"].copy()

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(locked_daily["date"], locked_daily["stress"], color="#f28c45", lw=1)
axes[0].set_ylabel("Stress 0–1")
axes[0].set_title("VKOSPI stress and defensive transfer")
axes[0].grid(alpha=.2)
axes[1].fill_between(locked_daily["date"], 100 * locked_daily["transfer_fraction"], color="#007a5e", alpha=.55)
axes[1].set_ylabel("Transfer (%)")
axes[1].grid(alpha=.2)
plt.tight_layout()
plt.show()

assert (locked_daily["signal_date"].dropna().to_numpy() < locked_daily.loc[locked_daily["signal_date"].notna(), "date"].to_numpy()).all()
print("Look-ahead 점검: 모든 신호일이 수익 발생일보다 앞섭니다.")


## 6. 비용·하위기간·불확실성

In [ ]:
costs = pd.read_csv(RESULTS / "vkospi_dynamic_cost_sensitivity.csv")
subperiods = pd.read_csv(RESULTS / "vkospi_dynamic_subperiods.csv")

display(Markdown("### 거래비용 민감도"))
display(costs.loc[:, ["Period", "Strategy", "CAGR", "Sharpe", "MDD", "TotalCost"]].style.format({
    "CAGR": "{:.2%}", "Sharpe": "{:.3f}", "MDD": "{:.2%}", "TotalCost": "{:.4f}"
}))
display(Markdown("### 하위기간"))
display(subperiods.loc[:, ["Period", "Strategy", "CAGR", "Sharpe", "MDD"]].style.format({
    "CAGR": "{:.2%}", "Sharpe": "{:.3f}", "MDD": "{:.2%}"
}))

bootstrap = report["locked"]["reconciled_multiobjective_bootstrap"]
display(Markdown("### 6개월 블록 부트스트랩 · 5,000회"))
display(pd.Series({
    "CAGR 개선 확률": bootstrap["probability_cagr_improves"],
    "Sharpe 개선 확률": bootstrap["probability_sharpe_improves"],
    "MDD 개선 확률": bootstrap["probability_mdd_improves"],
    "세 지표 동시 개선 확률": bootstrap["probability_all_three_improve"],
}).to_frame("probability").style.format("{:.1%}"))


## 7. Open Asset Pricing 연결과 해석 한계

[Chen–Zimmermann Signal Library](https://openassetpricing.com/SignalDoc-Browser.html)의 `betaVIX`, `RealizedVol`, `TrendFactor/Momentum`에서 **변동성 상태와 변화율을 과거 관측만으로 표현**하는 원칙을 참고했습니다. 다만 원 자료는 개별 주식 횡단면 수익 예측 신호이고, 이 노트북은 VKOSPI 시장 시계열로 자산군 노출을 조절하므로 직접 복제나 동일한 경제적 검정은 아닙니다.

Sharpe/MDD 개선의 부트스트랩 확률보다 CAGR 개선 확률이 낮습니다. 미래 수익 보장이나 실거래 체결 보장이 아니며, 세금·슬리피지·상품 추적오차·운용 제약은 별도 검토가 필요합니다.


## 8. 결과 묶음 저장

In [ ]:
output_zip = Path("/content/vkospi_dynamic_results.zip") if IN_COLAB else PROJECT_ROOT / "vkospi_dynamic_results.zip"
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.glob("vkospi_dynamic_*")):
        archive.write(path, arcname=path.name)
print("저장 완료:", output_zip)

if IN_COLAB:
    from google.colab import files
    files.download(str(output_zip))
